In [1]:
import torch
import torch.nn as nn

In [4]:
cfg=GPT_2_config={
  "vocab":50257,
  "context_len":1024,
  "emb_dim":768,
  "n_head":12,
  "n_layer":12,
  "dropout":0.1,
  "qkv_bias":False  
}

## Aproach-1 just using stacking of single casual attention


In [58]:
class Casual_Attention(nn.Module):
  def __init__(self,d_in,d_out,context_length,dropout,qkv_bias=False):
    super().__init__()
    
    self.w_query=nn.Linear(d_in, d_out, bias=qkv_bias) #layer not the actual wt
    self.w_key=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.w_value=nn.Linear(d_in, d_out,bias=qkv_bias)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))
  def forward(self,x):
    batch_size,num_token,d_in=x.shape
    
    queries=self.w_query(x)
    keys=self.w_key(x)
    value=self.w_value(x)
    
    atten_score=queries@keys.transpose(1,2)
    mask_bool=self.mask.bool()[:num_token,:num_token]
    atten_score.masked_fill(mask_bool,-torch.inf)  
    
    atten_weight=torch.softmax(atten_score/keys.shape[-1]**0.5,dim=-1)
    
    atten_weight=self.dropout(atten_weight)
    
    context_vector=atten_weight@value
    
    return context_vector
    

In [59]:
class MultiHeadAttentionWrapper(nn.Module):
  def __init__(self,d_in,d_out,context_length,dropout,num_head,qkv_bias=False):
    super().__init__()
    self.heads=nn.ModuleList(
      [Casual_Attention(d_in=d_in,d_out=d_out,context_length=context_length,dropout=dropout,qkv_bias=qkv_bias)
       for _ in range(num_head)
      ]
    )
  def forward(self,x):
    return torch.cat([head(x) for head in self.heads],dim=-1)  

In [66]:
torch.manual_seed(123)

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

batch = torch.stack((inputs, inputs), dim=0)

context_length = batch.shape[1]

d_in = batch.shape[2]
d_out = 2

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_head=4)

context_vecs = mha(batch)

context_vecs.shape

torch.Size([2, 6, 8])

In [67]:
for head in mha.heads:
  print(head)
  print("--------------------")

Casual_Attention(
  (w_query): Linear(in_features=3, out_features=2, bias=False)
  (w_key): Linear(in_features=3, out_features=2, bias=False)
  (w_value): Linear(in_features=3, out_features=2, bias=False)
  (dropout): Dropout(p=0.0, inplace=False)
)
--------------------
Casual_Attention(
  (w_query): Linear(in_features=3, out_features=2, bias=False)
  (w_key): Linear(in_features=3, out_features=2, bias=False)
  (w_value): Linear(in_features=3, out_features=2, bias=False)
  (dropout): Dropout(p=0.0, inplace=False)
)
--------------------
Casual_Attention(
  (w_query): Linear(in_features=3, out_features=2, bias=False)
  (w_key): Linear(in_features=3, out_features=2, bias=False)
  (w_value): Linear(in_features=3, out_features=2, bias=False)
  (dropout): Dropout(p=0.0, inplace=False)
)
--------------------
Casual_Attention(
  (w_query): Linear(in_features=3, out_features=2, bias=False)
  (w_key): Linear(in_features=3, out_features=2, bias=False)
  (w_value): Linear(in_features=3, out_featur

## Aproach-2 just using multi-head attention

In [47]:
class MultiHeadAttention(nn.Module):
  def __init__(self,d_in,d_out,context_length,num_heads,dropout,qkv_bias=False):
    super().__init__()
    
    self.d_in=d_in
    self.d_out=d_out
    self.num_head=num_heads
    if(num_heads==0):
      num_heads+=0.000000001
    self.head_dim=d_out//num_heads
    
    self.w_query=nn.Linear(d_in, d_out, bias=qkv_bias) #layer not the actual wt
    self.w_key=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.w_value=nn.Linear(d_in, d_out,bias=qkv_bias)
    self.out_proj=nn.Linear(d_out,d_out)
    self.context_length=context_length
    self.dropout=nn.Dropout(dropout)
    
    self.qkv_bias=qkv_bias
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))#register_buffer makes PyTorch officially aware of this tensor:here var is mask

  def forward(self,x):
    batch_size,num_token,d_in=x.shape
    
    keys=self.w_key(x)
    querys=self.w_query(x)
    values=self.w_value(x)
    
    #split using view    -- converting (batch_size,num_token,d_in) =-> (batch_size,num_token,num_head,head_dim)
    keys=keys.view(batch_size,num_token,self.num_head,self.head_dim)
    querys=querys.view(batch_size,num_token,self.num_head,self.head_dim)
    values=values.view(batch_size,num_token,self.num_head,self.head_dim)
    
    #taking the transpose and grouping according to the head  converting---> (batch_size,num_token,num_head,head_dim)=-> (batch_size,num_head,num_token,head_dim)
    keys=keys.transpose(1,2)
    querys=querys.transpose(1,2)
    values=values.transpose(1,2)
    
    atten_scores=querys@keys.transpose(2,3)
    
    mask_bool=self.mask.bool()[:num_token,:num_token]
    atten_scores.masked_fill(mask_bool,-torch.inf)
    
    attn_weigth=torch.softmax(atten_scores/keys.shape[-1]**0.5,dim=-1)
    attn_weigth=self.dropout(attn_weigth)
    
    context_vec=(attn_weigth@values).transpose(1,2)
    
    context_vec=context_vec.contiguous().view(batch_size,num_token,self.d_out)
    context_vec = self.out_proj(context_vec)
    
    return context_vec    

In [48]:
cfg=GPT_2_config={
  "vocab":50257,
  "context_len":1024,
  "emb_dim":768,
  "n_heads":12,
  "n_layer":12,
  "drop_rate":0.1,
  "qkv_bias":False  
}

In [49]:
attention = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_len"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )

In [ ]:
torch.manual_seed(123)

x = torch.rand(2, 3, 768)

context_vector = attention.forward(x)


print("Input Shape: ", x.shape)
print("Output Shape: ", context_vector.shape)
print(context_vector)